# DB Catalog Agent Check

This notebook validates `ai-agent-tools/agents/db_catalog_agent.py`.

The agent loads metadata **in-process** via `postgres_metadata.fetch_schema_catalog` (shared with `export_db_catalog.py`): `pg_catalog` DDL, `pg_get_functiondef` for routines, and an **`information_schema`** tally. Replies are parsed with **Pydantic** into a top-level `answer` string plus a `structured` object (`cited_table_names`, `cited_routine_names`, `missing_in_catalog`).

It will:
1. run the agent via `uv run`,
2. parse the agent JSON (including `structured`),
3. display answer, summary, and structured fields.
4. verify **SQL routine text** comes from catalog `procedures[].ddl` (not a Python file extractor); `export_db_catalog` CLI uses the same module.
5. run the agent on a **routine DDL** question and check grounding in `procedures[].ddl` (truncated in the LLM context to 2000 characters per object).

> Prerequisites:
> - project `.env` has PostgreSQL `PG*` (or `PG_DSN`) and GigaChat (`GIGACHAT_API_KEY` or `GIGACHAT_EMBEDDINGS_CREDENTIALS`) for steps 1–3 and 5.


In [19]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().parent
AGENT_PATH = PROJECT_ROOT / "ai-agent-tools" / "agents" / "db_catalog_agent.py"

print(f"Agent: {AGENT_PATH}")
assert AGENT_PATH.exists(), "Agent file not found"

Agent: /Users/nikolajabramov/PycharmProjects/Giga4DQM/ai-agent-tools/agents/db_catalog_agent.py


In [20]:
import os


def load_env(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


env_path = PROJECT_ROOT / ".env"
load_env(env_path)

if not (os.getenv("GIGACHAT_API_KEY") or os.getenv("GIGACHAT_EMBEDDINGS_CREDENTIALS")):
    raise RuntimeError(
        f"Missing GigaChat credentials in {env_path}. Add GIGACHAT_API_KEY or GIGACHAT_EMBEDDINGS_CREDENTIALS."
    )

schema = "s_grnplm_as_t_didsd_nnn_db_tmd"
question = "What does t_je_line table store and what are its key columns?"

cmd = [
    "uv", "run", "python", str(AGENT_PATH),
    "--schema", schema,
    "--question", question,
    "--pretty",
]

result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print("Exit code:", result.returncode)
if result.stderr.strip():
    print("STDERR:\n", result.stderr)

assert result.returncode == 0, "Agent execution failed"
assert result.stdout.strip(), "Agent returned empty output"

raw_output = result.stdout
print("Agent output captured.")

Exit code: 0
STDERR:
 /Users/nikolajabramov/PycharmProjects/Giga4DQM/.venv/lib/python3.12/site-packages/pydantic/main.py:1828: UserWarning: Field name "schema" in "ExportCatalogTool" shadows an attribute in parent "ValidatedFunction.create_model.<locals>.DecoratorBaseModel"
  return meta(
/Users/nikolajabramov/PycharmProjects/Giga4DQM/.venv/lib/python3.12/site-packages/pydantic/main.py:1828: UserWarning: Field name "schema" in "export_catalog_tool" shadows an attribute in parent "BaseModel"
  return meta(

Agent output captured.


In [21]:
payload = json.loads(raw_output)

print("Schema:", payload.get("schema"))
print("Question:", payload.get("question"))
print("Summary:", payload.get("summary"))

assert "structured" in payload, "agent should return structured (Pydantic) alongside answer"
st = payload["structured"]
for k in ("cited_table_names", "cited_routine_names", "missing_in_catalog"):
    assert k in st, f"structured missing key: {k}"
assert st.get("answer") == payload.get("answer"), "structured.answer must match top-level answer"

print("\nAnswer:\n")
print(payload.get("answer", ""))
print("\nStructured keys:", list(st.keys()))
print("cited_table_names:", st.get("cited_table_names"))
print("cited_routine_names:", st.get("cited_routine_names"))
print("missing_in_catalog:", st.get("missing_in_catalog"))

Schema: s_grnplm_as_t_didsd_nnn_db_tmd
Question: What does t_je_line table store and what are its key columns?
Summary: {'tables': 25, 'views': 0, 'procedures': 38}

Answer:

The table **t_je_line** stores staging journal entry lines used for debt calculation flows. Its key columns include:
- **int_org_id**: Internal organization identifier.
- **je_header_id**: Journal entry header identifier.
- **je_header_val_dt**: Journal entry value date.
- **je_line_cat_id**: Journal line category identifier.
- **je_line_coa_id**: Journal line account identifier.
- **je_line_corr_coa_id**: Corresponding account identifier.
- **je_line_cred_ind**: Credit or debit indicator.
- **je_line_local_amr**: Amount in local currency.
- **je_line_trans_amr**: Amount in transaction currency.
- **je_type_id**: Journal entry type identifier.

Structured keys: ['answer', 'cited_table_names', 'cited_routine_names', 'missing_in_catalog']
cited_table_names: ['t_je_line']
cited_routine_names: []
missing_in_catalog: [

## DB function / procedure code path

The graph always runs `export_catalog_tool` first: it opens **psycopg** and calls `postgres_metadata.fetch_schema_catalog` (same implementation as the `export_db_catalog` CLI, not a subprocess). Routine bodies come from `pg_get_functiondef`; table/view DDL from `pg_catalog`. For **Python** source snippets, `extract_function_code` in `tools.md` is a separate tool — the catalog agent does not use it for SQL.

The next cell checks (1) agent source wiring, (2) CLI export includes `procedures[].ddl` and an **`information_schema`** summary, (3) plausible `CREATE FUNCTION` / `CREATE PROCEDURE` DDL.

In [22]:
# DB routine definitions: export_catalog_tool → postgres_metadata.fetch_schema_catalog → procedures[].ddl.
# extract_function_code is Python-only and must not be used by the catalog agent for SQL routines.

_agent_src = AGENT_PATH.read_text(encoding="utf-8")
assert "export_catalog_tool" in _agent_src
assert "export_catalog_tool.invoke" in _agent_src, "fetch_catalog should invoke export_catalog_tool"
assert "fetch_schema_catalog" in _agent_src, "agent should call postgres_metadata.fetch_schema_catalog"
assert "extract_function_code" not in _agent_src, (
    "db_catalog_agent must not reference extract_function_code (Python-only; DB code is in catalog DDL)"
)

EXPORT_SCRIPT = PROJECT_ROOT / "ai-agent-tools" / "scripts" / "export_db_catalog.py"
assert EXPORT_SCRIPT.exists(), "export_db_catalog.py not found"

proc_cmd = [
    "uv",
    "run",
    "python",
    str(EXPORT_SCRIPT),
    "--schema",
    schema,
    "--sections",
    "procedures",
]
proc_run = subprocess.run(proc_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print("export_db_catalog (procedures) exit code:", proc_run.returncode)
if proc_run.stderr.strip():
    print("STDERR:\n", proc_run.stderr)
assert proc_run.returncode == 0, f"export procedures failed: {proc_run.stderr or proc_run.stdout}"

proc_cat = json.loads(proc_run.stdout)
assert "information_schema" in proc_cat, "export should include information_schema tallies"
print("information_schema:", proc_cat.get("information_schema"))
procs = proc_cat.get("procedures", [])
assert procs, "expected at least one procedure in catalog for this schema"
p0 = procs[0]
ddl = (p0.get("ddl") or "").upper()
assert "CREATE" in ddl and ("FUNCTION" in ddl or "PROCEDURE" in ddl), (
    f"expected CREATE FUNCTION/PROCEDURE DDL; got: {p0.get('ddl', '')[:200]!r}"
)

print("DB routine code path OK: export_catalog_tool → procedures[].ddl")
print("Sample:", p0.get("kind"), p0.get("signature", p0.get("name")), "ddl chars:", len(p0.get("ddl") or ""))

export_db_catalog (procedures) exit code: 0
information_schema: {'tables_by_type': {'BASE TABLE': 25}, 'routine_count': 38}
DB routine code path OK: export_catalog_tool → procedures[].ddl
Sample: function add_arr_log(in_workflow2_run_id bigint, strobject text, param text, sql_query text, INOUT vlog s_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log[]) ddl chars: 458


## Agent: extracting DB function / procedure code

The LLM still sees each routine’s `ddl` truncated to **2000 characters** in the prompt. The model must return **JSON** matching `CatalogAnswer` (see `PydanticOutputParser` in `db_catalog_agent.py`); the CLI exposes the main text as `answer` and the full object as `structured`.

The cell below picks a routine, asks for the exact DDL prefix, checks **grounding** in the top-level `answer`, and checks **`structured`** matches and lists expected keys.

In [23]:
# Reuse procedures export; pick a routine with substantive DDL (same as agent's procedures list).
# Agent truncates each ddl to 2000 chars when building the GigaChat prompt.
DDL_CONTEXT_LIMIT = 2000
EXPORT_SCRIPT = PROJECT_ROOT / "ai-agent-tools" / "scripts" / "export_db_catalog.py"

_proc_cmd = [
    "uv",
    "run",
    "python",
    str(EXPORT_SCRIPT),
    "--schema",
    schema,
    "--sections",
    "procedures",
]
_pr = subprocess.run(_proc_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
assert _pr.returncode == 0, _pr.stderr
_proc_cat = json.loads(_pr.stdout)
_candidates = [p for p in _proc_cat.get("procedures", []) if len(p.get("ddl") or "") >= 200]
assert _candidates, "need at least one procedure with DDL for grounding check"
_target = _candidates[0]
_rname = _target.get("name") or ""
_ddl_full = _target.get("ddl") or ""
_ddl_in_prompt = _ddl_full[:DDL_CONTEXT_LIMIT]

# Consecutive prefix of catalog DDL the model receives (used as grounding needle for the answer).
_chunk_len = 120
_needle = _ddl_in_prompt[:_chunk_len]
if len(_needle) < 80:
    raise AssertionError("DDL prefix too short for a meaningful needle")

_q = (
    f"In the catalog context, the routine `{_rname}` has a `ddl` field. "
    f"Copy the **exact** beginning of that DDL (at least the `CREATE` line and the following text) as it appears in the context — do not paraphrase."
)
_dbfx_cmd = [
    "uv",
    "run",
    "python",
    str(AGENT_PATH),
    "--schema",
    schema,
    "--question",
    _q,
    "--pretty",
]
_dbfx = subprocess.run(_dbfx_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print("db function extraction agent exit code:", _dbfx.returncode)
if _dbfx.stderr.strip():
    print("STDERR:\n", _dbfx.stderr)
assert _dbfx.returncode == 0, "agent failed (DB function DDL question)"
fx_payload = json.loads(_dbfx.stdout)
assert "structured" in fx_payload
_fx = fx_payload["structured"]
assert _fx.get("answer") == fx_payload.get("answer")
for _k in ("cited_table_names", "cited_routine_names", "missing_in_catalog"):
    assert _k in _fx
_ans = fx_payload.get("answer") or ""
# Must mention the routine; answer should contain verbatim catalog text
assert _rname in _ans.replace("`", ""), f"expected routine name in answer: {_rname!r}"
if _needle not in _ans:
    # allow minor line-ending/space diffs: normalize spaces for a shorter needle
    import re as _re

    _n2 = _re.sub(r"\s+", " ", _needle).strip()
    _a2 = _re.sub(r"\s+", " ", _ans)
    assert _n2 in _a2, (
        "answer not grounded in catalog DDL prefix — expected a verbatim substring of procedures[].ddl "
        f"(first {_chunk_len} chars). Answer excerpt: {_ans[:500]!r}"
    )
print("DB function code extraction OK: answer grounded in catalog DDL prefix for", repr(_rname))
print("Answer excerpt:", repr(_ans[:400]))

db function extraction agent exit code: 0
STDERR:
 /Users/nikolajabramov/PycharmProjects/Giga4DQM/.venv/lib/python3.12/site-packages/pydantic/main.py:1828: UserWarning: Field name "schema" in "ExportCatalogTool" shadows an attribute in parent "ValidatedFunction.create_model.<locals>.DecoratorBaseModel"
  return meta(
/Users/nikolajabramov/PycharmProjects/Giga4DQM/.venv/lib/python3.12/site-packages/pydantic/main.py:1828: UserWarning: Field name "schema" in "export_catalog_tool" shadows an attribute in parent "BaseModel"
  return meta(



AssertionError: answer not grounded in catalog DDL prefix — expected a verbatim substring of procedures[].ddl (first 120 chars). Answer excerpt: 'CREATE OR REPLACE FUNCTION s_grnplm_as_t_didsd_nnn_db_tmd.add_arr_log\nin_workflow2_run_id bigint, strobject text, param text, sql_query text, INOUT vlog s_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log[])\nRETURNS s_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log[]\nLANGUAGE plpgsql\nAS $function$\n\nbegin\n\n vlog = array_append(vlog, row(in_workflow2_run_id, clock_timestamp()::text, strobject, sql_query, param)::s_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log) ;\n\nend; \n\n$function$'

In [ ]:
# Optional: run additional questions
extra_questions = [
    "List tables related to agr_cred and explain their purpose.",
    "Are there views in this schema?",
    "Which routine calculates main debts?",
]

for q in extra_questions:
    cmd = [
        "uv", "run", "python", str(AGENT_PATH),
        "--schema", schema,
        "--question", q,
    ]
    r = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"Question failed: {q}")
        print(r.stderr)
        continue
    p = json.loads(r.stdout)
    print("-" * 80)
    print("Q:", q)
    print("A:", p.get("answer", "")[:600], "..." if len(p.get("answer", "")) > 600 else "")
    if p.get("structured"):
        st8 = p["structured"]
        assert st8.get("answer") == p.get("answer")
        print(
            "  structured: cited_tables=",
            len(st8.get("cited_table_names") or []),
            "cited_routines=",
            len(st8.get("cited_routine_names") or []),
        )

--------------------------------------------------------------------------------
Q: List tables related to agr_cred and explain their purpose.
A: ### Tables Relatedto agr_cred

1. **d_agr_cred**
   - **Description:** Compact agreement-credit table used in local bootstrap mode.
   - **Columns:** agr_cred_id, prnt_agr_cred_id, info_system_id

2. **d_agr_cred_1497470240**
   - **Description:** No description provided.
   - **Columns:** agr_cred_id, prnt_agr_cred_id, info_system_id

3. **d_agr_cred_coa**
   - **Description:** Agreement-account bindings with account metadata.
   - **Columns:** Includes agr_id, prnt_agr_cred_id, sl_coa_id, meas_cd, start_dt, end_dt, etc.

4. **d_agr_cred_coa_1497470240**
   - **Description:** No description p ...
--------------------------------------------------------------------------------
Q: Are there views in this schema?
A: Based on the provided catalog context, there are no views in the schema `s_grnplm_as_t_didsd_nnn_db_tmd`. The `views` key in the c